In [ ]:
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
from matplotlib.pylab import rcParams
from statsmodels.tsa.stattools import adfuller
from pathlib import Path
import pmdarima as pm
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error 
import csv

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

windspeed_df = pd.read_csv(str(ROOT / "data/DMI_ws.csv"))
irr_df = pd.read_csv(str(ROOT / "data/DMI_radiation.csv"))
PV_df = pd.read_csv(str(ROOT / "data/SyslabPV_15min_collective_PV.csv"))
PV_df["ts"] = pd.to_datetime(PV_df['ts'])
PV_df = PV_df.set_index("ts")
PV_df = PV_df
Wind_df = pd.read_csv(str(ROOT / "data/SyslabWind_15min_nozeros.csv"))
df = pd.read_csv(str(ROOT / "data/combined_data_positive_gen.csv"))

In [ ]:
seasonality = 24*4 
PV_df["Power diff"] = PV_df['Collective PV'].diff(periods=seasonality)

In [ ]:
result = seasonal_decompose(PV_df['Collective PV'].dropna(), model='addiditve', period=seasonality)
trend = result.trend.dropna()
seasonal = result.seasonal.dropna()
residual = result.resid.dropna()

In [ ]:

# Plot the decomposed components
plt.figure(figsize=(6,6))
#x_labels = [t.strftime('%m %Y') for t in PV_df.index]
plt.subplot(4, 1, 1)
plt.plot(PV_df.index,PV_df['Collective PV'], label='Original Series')

#plt.xticks(range(0, PV_df.size, 2880), x_labels[::2880], rotation=45)
plt.legend()

plt.subplot(4, 1, 2)
plt.plot(trend, label='Trend')
plt.legend()

plt.subplot(4, 1, 3)
plt.plot(seasonal, label='Seasonal')
plt.legend()

plt.subplot(4, 1, 4)
plt.plot(residual, label='Residuals')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
from matplotlib.pylab import rcParams
from statsmodels.tsa.stattools import adfuller
from pathlib import Path
import pmdarima as pm
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

In [ ]:
def plot_predicted_vs_measured(measured,predicted,dataset,exos,exo_names,plot=True):

    r2 = r2_score(measured, predicted)
    mae = mean_absolute_error(measured, predicted)
    rmse = mean_squared_error(measured, predicted) ** 0.5
    print("R²:", r2)
    
    if plot: # Scatter plot: measured vs predicted on test set
        
        plt.figure(figsize=(6, 6))
        plt.scatter(measured, predicted, alpha=0.6)
        plt.xlabel("Measured power")
        plt.ylabel("Predicted power")
        plt.title(f"{dataset} model: measured vs predicted (test set)")

        min_val = min(measured.min(), predicted.min())
        max_val = max(measured.max(), predicted.max())
        plt.plot([min_val, max_val], [min_val, max_val])

        plt.show()
        for i in range(len(exo_names)):
            exo_name = exo_names[i]
            exo = exos[exo_name]
            plt.figure(figsize=(6, 6))
            plt.scatter(exo, predicted, alpha=0.6)
            plt.xlabel(f"{exo_name}")
            plt.ylabel("Predicted power")
            plt.title(f"{dataset} model: {exo_name} vs predicted (on test set)")

            
            plt.show()

    
    
    return r2, mae, rmse

In [ ]:
def Sarimax(Type,file_path,endo_name,exo_names:list,granularity=10,cut_in = 1 ,order = (1,0,1),seasonal_order = (1,1,1),gaps = (["2025-06-14 00:00:00","2025-06-21 00:00:00"]),save_new_file = False,filename="",plot = True):
    """Using the SARIMAX function to fill gaps in the endogenous variable using the exogeneous variable
    Granularity is the amount of time between measurements in minutes
    Type is either "Wind" or "Solar" """
    exo_name = exo_names[0]
    try:
        df = pd.read_csv(file_path,delimiter=",")
    except Exception as e:
        print(f"Error reading file: {e}")
        return
    df["ts"] = pd.to_datetime(df["ts"])
    df = df.set_index("ts") # Set datetime as index
    df = df.loc[df.index >"2025-05-14 07:00:00"] # start from when our power data starts
    df = df.asfreq('30min')
    gap_mask = np.logical_or.reduce([(start < df.index) & (df.index < end) for start, end in gaps])

    
    print("Gaps make up ",100*gap_mask.sum()/(len(df)-df[endo_name].isna().sum()),"% of the data")

    for ex in exo_names:
        df[ex] = df[ex].interpolate() # REMOVE ANY NANS

    df_modelled = df.copy()
    
    if Type == "Wind":
        # Wind power is proportional to the cubic wind speed
        df.loc[df[exo_name] < cut_in, endo_name] = 0 # set low wind periods to zero

        gap_measured = df.loc[gap_mask,endo_name] # save the data that we will later use for testing

        gap_mask_2 = gap_mask[df[exo_name] >= cut_in]
        df = df.loc[df[exo_name] >= cut_in] # remove low wind periods to reduce how much it gets skewed
        df.loc[gap_mask_2,endo_name] = np.nan

        df["active"] = (df[exo_name]>= 3).astype(int)
        df["w3"] = df[exo_name]**3
        
        endo = df[endo_name]
        exo = df[[*exo_names,"w3"]]
        print("Preparing Model for Wind Turbine")

    elif Type == "Solar":
        df.loc[df[exo_name] < cut_in, endo_name] = 0 # set night time to zero

        gap_measured = df.loc[gap_mask,endo_name] # save the data that we will later use for testing

        gap_mask_2 = gap_mask[df[exo_name] >= cut_in]
        df = df.loc[df[exo_name] >= cut_in] # remove night time to reduce how much it gets skewed
        df.loc[gap_mask_2,endo_name] = np.nan

        endo = df[endo_name]
        exo = df[exo_names] 
        print("Preparing Model for PV unit")
    

    if gap_measured.isna().sum().astype(int) != 0: 
        print("Gap contains NaNs so the model can not be verified. Retry with different gap.")
        return 

    fraction_dayvsnight = len(df)/len(df_modelled)
    seasonality = 24*60/granularity*fraction_dayvsnight ####### Takes the average time between the time today and the same time tomorrow (complex since we removed nighttime values)
    seasonal_order_full = (*seasonal_order, int(seasonality))

    model = SARIMAX(
        endo,
        exog=exo,
        order=order,
        seasonal_order=seasonal_order_full,  # daily seasonality
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    results = model.fit()
    
    print("The AIC is: ", results.aic)
    endo_modelled = results.get_prediction().predicted_mean # USES KALMAN SMOOTHING ( includes both past and future values to do "predictions")
    df["endo_filled"] = df[endo_name]
    df.loc[endo.isna(),"endo_filled"] = endo_modelled.loc[endo.isna()]



    df_modelled.loc[df_modelled[exo_name] >= cut_in, endo_name +"modelled"] = df["endo_filled"] # redo the zero rows that were removed because low windspeed
    df_modelled.loc[df_modelled[exo_name] < cut_in, endo_name+ "modelled"] = 0 # set everything 0 with no windspeed
    df_modelled.loc[df_modelled[endo_name + "modelled"] < 0,endo_name + "modelled" ]= 0

    if save_new_file:
        file_name = f'{filename}.csv'
        df_modelled.to_csv(file_name, index=True)


    
    gap_predicted = df_modelled.loc[gap_mask,endo_name + "modelled"]
    gap_exo = df_modelled.loc[gap_mask,exo_names]
    return plot_predicted_vs_measured(gap_measured,gap_predicted,endo_name,gap_exo,exo_names,plot), float(round(results.aic,2)), float(round(results.bic,2))



In [ ]:
combined_data = str(ROOT / "data/combined_data_weather_and_power_30min.csv")

In [ ]:
gaps = (["2025-06-19 00:00:00","2025-06-29 00:00:00"],
        ["2025-07-05 00:00:00","2025-07-10 00:10:00"],
        ["2025-10-05 13:30:00","2025-10-07 08:00:00"],
        ["2025-12-10 23:30:00","2025-12-11 06:30:00"],
        ["2025-12-23 23:30:00","2025-12-24 18:30:00"],
        ["2025-05-23 17:30:00","2025-05-26 09:30:00"],
        ["2025-06-09 16:30:00","2025-06-12 05:00:00"],
        ["2025-08-27 16:30:00","2025-08-30 05:00:00"],
        ["2025-11-09 16:30:00","2025-11-13 03:30:00"],
        ["2025-11-20 12:00:00","2025-11-22 14:30:00"],
        ["2025-07-01 13:30:00","2025-07-02 02:00:00"],
        ["2025-07-15 13:30:00","2025-07-20 02:00:00"],
        ["2025-09-01 16:30:00","2025-09-06 08:00:00"],
        ["2025-09-10 11:30:00","2025-09-13 10:00:00"],
        ["2026-03-15 23:30:00","2026-03-18 17:30:00"])
Sarimax("Wind",combined_data,"Aircon_WT Power",["wind_speed","wind_max"],cut_in=1,granularity = 30,order = (0,0,0),
                                seasonal_order = (0,0,0),save_new_file=False, filename="",
                                gaps = gaps,plot = False)

In [ ]:
gaps = (["2025-06-19 00:00:00","2025-06-29 00:00:00"],
        ["2025-07-05 00:00:00","2025-07-10 00:10:00"],
        ["2025-10-05 13:30:00","2025-10-07 08:00:00"],
        ["2025-12-10 23:30:00","2025-12-11 06:30:00"],
        ["2025-12-23 23:30:00","2025-12-24 18:30:00"],
        ["2025-05-23 17:30:00","2025-05-26 09:30:00"],
        ["2025-06-09 16:30:00","2025-06-12 05:00:00"],
        ["2025-08-27 16:30:00","2025-08-30 05:00:00"],
        ["2025-11-09 16:30:00","2025-11-13 03:30:00"],
        ["2025-11-20 12:00:00","2025-11-22 14:30:00"],
        ["2025-07-01 13:30:00","2025-07-02 02:00:00"],
        ["2025-07-15 13:30:00","2025-07-20 02:00:00"],
        ["2025-09-01 16:30:00","2025-09-06 08:00:00"],
        ["2025-09-10 11:30:00","2025-09-13 10:00:00"],
        ["2026-03-15 23:30:00","2026-03-18 17:30:00"])
Sarimax("Solar",combined_data,"B715",["radia_glob","temp_dry"],granularity = 30,cut_in=-1,order = (0,0,0),
                                seasonal_order = (0,0,0),save_new_file=False, filename="PV715 Modelled Data.csv",
                                gaps = gaps,plot = False)

In [ ]:
orders = [(0,0,0),(1,0,0),(0,1,0),(0,0,1),(1,1,0),(1,0,1),(0,1,1),(1,1,1)]
seasonal_orders = [(0,0,0),(1,0,0),(0,1,0),(0,0,1),(1,1,0),(1,0,1),(0,1,1),(1,1,1)]

gaps = (["2025-06-19 00:00:00","2025-06-29 00:00:00"],
        ["2025-07-05 00:00:00","2025-07-10 00:10:00"],
        ["2025-10-05 13:30:00","2025-10-07 08:00:00"],
        ["2025-12-10 23:30:00","2025-12-11 06:30:00"],
        ["2025-12-23 23:30:00","2025-12-24 18:30:00"],
        ["2025-05-23 17:30:00","2025-05-26 09:30:00"],
        ["2025-06-09 16:30:00","2025-06-12 05:00:00"],
        ["2025-08-27 16:30:00","2025-08-30 05:00:00"],
        ["2025-11-09 16:30:00","2025-11-13 03:30:00"],
        ["2025-11-20 12:00:00","2025-11-22 14:30:00"],
        ["2025-07-01 13:30:00","2025-07-02 02:00:00"],
        ["2025-07-15 13:30:00","2025-07-20 02:00:00"],
        ["2025-09-01 16:30:00","2025-09-06 08:00:00"],
        ["2025-09-10 11:30:00","2025-09-13 10:00:00"],
        ["2026-03-15 23:30:00","2026-03-18 17:30:00"])
params_list = [["radia_glob","temp_dry"],["radia_glob","temp_dry"]]
filenames = ['PV_power_AIC_BIC_0cutin.csv','PV_power_AIC_BIC_1cutin.csv']
types = ["Solar","Solar"]
units = ["B715","B715"]
cutins = [-1,1]

i = 1

params = params_list[i]
unit = units[i]
filename = f"../data/SARIMAX Analysis/{filenames[i]}"
type = types[i]
cutin = cutins[i]
with open(filename, 'w') as csvfile: 
        #rowwriter = csv.writer(csvfile, delimiter=',')
        rowwriter.writerow(['order', 'seasonal_order', 'r2',"mae","rmse","AIC","BIC"])
        print(f"Starting the evaluation of {unit} with cutin={cutin} will be saved in {filename}")
        for order in orders: 
                for seasonal_order in seasonal_orders:
                        (r2, mae, rmse), AIC, BIC = Sarimax(type,
                                                combined_data,
                                                unit,
                                params,cut_in=cutin,
                                granularity = 30,
                                order = order,
                                seasonal_order = seasonal_order,
                                save_new_file=False,
                                filename="",
                                gaps = gaps,
                                plot = False)
                        rowwriter.writerow([order,seasonal_order,round(r2,3),round(mae,3),round(rmse,3),AIC,BIC])
                        csvfile.flush()
                print(f"Order {order} is completed")


In [ ]:
orders = [(0,0,0),(1,0,0),(0,1,0),(0,0,1),(1,1,0),(1,0,1),(0,1,1),(1,1,1)]
seasonal_orders = [(0,0,0),(1,0,0),(0,1,0),(0,0,1),(1,1,0),(1,0,1),(0,1,1),(1,1,1)]

gaps = (["2025-06-19 00:00:00","2025-06-29 00:00:00"],
        ["2025-07-05 00:00:00","2025-07-10 00:10:00"],
        ["2025-10-05 13:30:00","2025-10-07 08:00:00"],
        ["2025-12-10 23:30:00","2025-12-11 06:30:00"],
        ["2025-12-23 23:30:00","2025-12-24 18:30:00"],
        ["2025-05-23 17:30:00","2025-05-26 09:30:00"],
        ["2025-06-09 16:30:00","2025-06-12 05:00:00"],
        ["2025-08-27 16:30:00","2025-08-30 05:00:00"],
        ["2025-11-09 16:30:00","2025-11-13 03:30:00"],
        ["2025-11-20 12:00:00","2025-11-22 14:30:00"],
        ["2025-07-01 13:30:00","2025-07-02 02:00:00"],
        ["2025-07-15 13:30:00","2025-07-20 02:00:00"],
        ["2025-09-01 16:30:00","2025-09-06 08:00:00"],
        ["2025-09-10 11:30:00","2025-09-13 10:00:00"],
        ["2026-03-15 23:30:00","2026-03-18 17:30:00"])
params_list = [["radia_glob","temp_dry"],["radia_glob","temp_dry"]]
filenames = ['PV_power_AIC_BIC_0cutin.csv','PV_power_AIC_BIC_1cutin.csv']
types = ["Solar","Solar"]
units = ["B715","B715"]
cutins = [-1,1]
for i in range(4):
        params = params_list[i]
        unit = units[i]
        filename = f"../data/SARIMAX Analysis/{filenames[i]}"
        type = types[i]
        cutin = cutins[i]
        #with open(filename, 'w') as csvfile: 
                rowwriter = csv.writer(csvfile, delimiter=',')
                rowwriter.writerow(['order', 'seasonal_order', 'r2',"mae","rmse","AIC","BIC"])
                for order in orders: 
                        for seasonal_order in seasonal_orders:
                                (r2, mae, rmse), AIC, BIC = Sarimax(type,
                                                       combined_data,
                                                       unit,
                                        params,cut_in=cutin,
                                        granularity = 30,
                                        order = order,
                                        seasonal_order = seasonal_order,
                                        save_new_file=False,
                                        filename="",
                                        gaps = gaps,
                                        plot = False)
                                rowwriter.writerow([order,seasonal_order,round(r2,3),round(mae,3),round(rmse,3),AIC,BIC])
                                csvfile.flush()
                        print(f"Order {order} is completed")
        print(f"Evaluation of {unit} with cutin={cutin} is completed and saved in {filename}")
                

In [ ]:
orders = [(0,0,0),(1,0,0),(0,1,0),(0,0,1),(1,1,0),(1,0,1),(0,1,1),(1,1,1)]
seasonal_orders = [(0,0,0),(1,0,0),(0,1,0),(0,0,1),(1,1,0),(1,0,1),(0,1,1),(1,1,1)]
gaps = (["2025-06-19 00:00:00","2025-06-29 00:00:00"],
        ["2025-07-05 00:00:00","2025-07-10 00:10:00"],
        ["2025-10-05 13:30:00","2025-10-07 08:00:00"],
        ["2025-12-10 23:30:00","2025-12-11 06:30:00"],
        ["2025-12-23 23:30:00","2025-12-24 18:30:00"],
        ["2025-05-23 17:30:00","2025-05-26 09:30:00"],
        ["2025-06-09 16:30:00","2025-06-12 05:00:00"],
        ["2025-08-27 16:30:00","2025-08-30 05:00:00"],
        ["2025-11-09 16:30:00","2025-11-13 03:30:00"],
        ["2025-11-20 12:00:00","2025-11-22 14:30:00"],
        ["2025-07-01 13:30:00","2025-07-02 02:00:00"],
        ["2025-07-15 13:30:00","2025-07-20 02:00:00"],
        ["2025-09-01 16:30:00","2025-09-06 08:00:00"],
        ["2025-09-10 11:30:00","2025-09-13 10:00:00"],
        ["2026-03-15 23:30:00","2026-03-18 17:30:00"])
params = ["radia_glob","temp_dry"]
dic_w = {}
with open('PV_power_R2.csv', 'w') as csvfile: 
        rowwriter = csv.writer(csvfile, delimiter=',')
        rowwriter.writerow(['order', 'seasonal_order', 'r2',"AIC","BIC"])
        for order in orders: 
                for seasonal_order in seasonal_orders:
                        r2 = Sarimax("Solar",combined_data,"B715",params,granularity = 30,order = order,
                                seasonal_order = seasonal_order,save_new_file=False, filename="",
                                gap = gap,plot = False)
                        rowwriter.writerow([order,seasonal_order,r2])
                        csvfile.flush()
                print(f"Order {order} is completed")

In [ ]:
orders = [(0,0,0),(0,0,1),(0,0,2),(1,0,0),(2,0,0),(1,0,1)]
seasonal_orders = [(0,0,0),(0,0,1),(0,1,1),(1,0,0),(1,1,0),(1,0,1),(1,1,1),(0,1,0)]
gap = ["2025-06-19 00:00:00","2025-06-29 00:00:00"]
params = ["radia_glob","temp_dry"]
dic_w = {}
with open('PV_power_R2_0cutin.csv', 'w') as csvfile: 
        rowwriter = csv.writer(csvfile, delimiter=',')
        rowwriter.writerow(['order', 'seasonal_order', 'r2'])
        for order in orders: 
                for seasonal_order in seasonal_orders:
                        r2 = Sarimax("Solar",combined_data,"B715",params,cut_in=-1,granularity = 30,order = order,
                                seasonal_order = seasonal_order,save_new_file=False, filename="",
                                gap = gap,plot = False)
                        rowwriter.writerow([order,seasonal_order,r2])
                        csvfile.flush()
                print(f"Order {order} is completed")

In [ ]:
orders = [(0,0,0),(0,0,1),(0,0,2),(1,0,0),(2,0,0),(1,0,1)]
seasonal_orders = [(0,0,0),(0,0,1),(0,1,1),(1,0,0),(1,1,0),(1,0,1),(1,1,1),(0,1,0)]
gap = ["2025-06-19 00:00:00","2025-06-29 00:00:00"]
params = ["wind_speed","wind_max"]
with open('Windpower_R2_05cutin.csv', 'w') as csvfile: 
        rowwriter = csv.writer(csvfile, delimiter=',')
        rowwriter.writerow(['order', 'seasonal_order', 'r2'])
        for order in orders: 
                for seasonal_order in seasonal_orders:
                        r2 = Sarimax("Wind",combined_data,"Aircon_WT Power",params,cut_in=0.5,granularity = 30,order = order,
                                seasonal_order = seasonal_order,save_new_file=False, filename="",
                                gap = gap,plot = False)
                        rowwriter.writerow([order,seasonal_order,r2])
                        csvfile.flush()
                print(f"Order {order} is completed")
                